# Phase 2: Country stories and Kaya Score

This notebook:

1. Loads `data/processed/kaya_dataset.csv`
2. Decomposes CO₂ change for the US, China, Sweden, and France
3. Applies the **locked** Kaya Champion score (see `SCORING.md` / `src/kaya_score.py`)

Run the Phase 1 pipeline first if the processed dataset is missing.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "src"))

from kaya_score import (
    ScoreConfig,
    load_kaya_dataset,
    log_kaya_decomposition,
    score_countries,
)

df = load_kaya_dataset(ROOT / "data" / "processed" / "kaya_dataset.csv")
df.head()


## Country stories (log Kaya decomposition)

For each country, change in \(\ln CO_2\) equals the sum of changes in \(\ln\) of population, GDP per capita, energy intensity, and carbon intensity.


In [ ]:
FOCUS = {
    "USA": "United States",
    "CHN": "China",
    "SWE": "Sweden",
    "FRA": "France",
}
START, END = 2000, 2022


def window_rows(iso: str, start: int = START, end: int = END):
    g = df[df["iso_code"] == iso].sort_values("year")
    s = g[g["year"] == start]
    e = g[g["year"] == end]
    if s.empty or e.empty:
        raise ValueError(f"Missing {iso} for {start} or {end}")
    return s.iloc[0], e.iloc[0]


def pct(a, b):
    return 100 * (b - a) / a


stories = []
for iso, name in FOCUS.items():
    s, e = window_rows(iso)
    parts = log_kaya_decomposition(s, e)
    stories.append({
        "country": name,
        "iso": iso,
        "co2_pct": pct(s.co2, e.co2),
        "pop_pct": pct(s.population, e.population),
        "gdp_pc_pct": pct(s.gdp_per_capita, e.gdp_per_capita),
        "ei_pct": pct(s.energy_intensity, e.energy_intensity),
        "ci_pct": pct(s.carbon_intensity, e.carbon_intensity),
        **parts,
    })

stories_df = pd.DataFrame(stories)
stories_df.round(3)


In [ ]:
# Bar chart: log contributions to CO2 change
fig, axes = plt.subplots(2, 2, figsize=(10, 8), sharey=True)
factors = [
    ("dln_population", "Population"),
    ("dln_gdp_per_capita", "GDP/capita"),
    ("dln_energy_intensity", "Energy intensity"),
    ("dln_carbon_intensity", "Carbon intensity"),
]
colors = ["#4C78A8", "#F58518", "#54A24B", "#E45756"]

for ax, (_, row) in zip(axes.ravel(), stories_df.iterrows()):
    vals = [row[k] for k, _ in factors]
    ax.barh([lab for _, lab in factors], vals, color=colors)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_title(f"{row['country']} ({START}–{END})\nΔln CO₂ = {row['dln_co2']:.3f}")
    ax.set_xlabel("Δ ln")

fig.suptitle("Kaya log decomposition: what drove CO₂ change?", y=1.02)
fig.tight_layout()
plt.show()


### Narrative takeaways

- **United States:** population and GDP/capita up; energy and carbon intensity down enough that total CO₂ falls vs 2000 (and further vs mid-2000s peak).
- **China:** huge prosperity growth dominates; intensity improvements help but do not offset scale → CO₂ rises sharply.
- **Sweden / France:** emissions fall with intensity improvements; prosperity still rises (decoupling).

These are descriptive decompositions, not single-cause claims.


## Kaya Champion score

Specification locked in `SCORING.md`. Weights: decarbonization 30%, prosperity 25%, efficiency 20%, clean energy 25%.


In [ ]:
scored = score_countries(df)
print(f"Eligible countries: {len(scored)} (window {scored.start_year.iloc[0]}→ typically {scored.end_year.mode().iloc[0]})")
scored.head(15)[
    [
        "country",
        "kaya_score",
        "score_decarbonization",
        "score_prosperity",
        "score_efficiency",
        "score_clean",
        "co2_pct",
        "gdp_per_capita_pct",
    ]
].round(2)


In [ ]:
focus_iso = list(FOCUS)
focus_scores = scored[scored["iso_code"].isin(focus_iso)].copy()
focus_scores[["country", "kaya_score", "co2_pct", "gdp_per_capita_pct", "energy_intensity_pct", "carbon_intensity_pct"]].round(3)


In [ ]:
# Top / bottom snapshot
fig, ax = plt.subplots(figsize=(8, 6))
top = scored.head(12).iloc[::-1]
ax.barh(top["country"], top["kaya_score"], color="#4C78A8")
ax.set_xlabel("Kaya score (0–100)")
ax.set_title("Top Kaya Champion scores (2000 → latest)")
ax.set_xlim(0, 100)
fig.tight_layout()
plt.show()


## Method notes

- Clip bounds and weights are **product choices** for v1, not physical constants.
- Transition economies often rank high — real post-2000 efficiency + growth; explain in UI.
- Do not ship a leaderboard UI until this notebook and `SCORING.md` match production code (`src/kaya_score.py`).
